In [1]:
#!/usr/bin/env python3
"""
LLAMA ROBUSTNESS TESTING
========================
Comprehensive statistical tests to determine if Llama's superior performance
is genuine or a statistical fluke.

Tests included:
1. Statistical significance testing (bootstrap & permutation tests)
2. Cross-validation stability analysis
3. Robustness checks across subgroups
4. Error pattern analysis
5. Consistency checks with different random seeds
6. Performance degradation tests
"""

# ================================================================
# Fix matplotlib compatibility
# ================================================================
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, ttest_rel, wilcoxon
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.preprocessing import StandardScaler

# Set style
sns.set_style("whitegrid")

print("="*80)
print("LLAMA PERFORMANCE VALIDATION & ROBUSTNESS TESTING")
print("="*80)

# ================================================================
# 1. Load Data
# ================================================================

print("\n" + "="*80)
print("1. LOADING DATA")
print("="*80)

df = pd.read_csv("predictions_all_stages_long.csv")
model_cols = [col for col in df.columns if col.startswith('pred_')]
df['pred_ensemble'] = df[model_cols].mean(axis=1)

# Add continent mapping
continent_mapping = {
    'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'Argentina': 'South America',
    'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europe', 'Bangladesh': 'Asia',
    'Belgium': 'Europe', 'Benin': 'Africa', 'Bolivia': 'South America', 'Bosnia Herzegovina': 'Europe',
    'Botswana': 'Africa', 'Brazil': 'South America', 'Bulgaria': 'Europe', 'Burkina Faso': 'Africa',
    'Cambodia': 'Asia', 'Cameroon': 'Africa', 'Canada': 'North America', 'Chad': 'Africa',
    'Chile': 'South America', 'China': 'Asia', 'Colombia': 'South America', 'Congo Brazzaville': 'Africa',
    'Costa Rica': 'North America', 'Croatia': 'Europe', 'Cyprus': 'Europe', 'Czech Republic': 'Europe',
    'Denmark': 'Europe', 'Dominican Republic': 'North America', 'Ecuador': 'South America', 'Egypt': 'Africa',
    'El Salvador': 'North America', 'Estonia': 'Europe', 'Ethiopia': 'Africa', 'Finland': 'Europe',
    'France': 'Europe', 'Gabon': 'Africa', 'Georgia': 'Asia', 'Germany': 'Europe',
    'Ghana': 'Africa', 'Greece': 'Europe', 'Guatemala': 'North America', 'Guinea': 'Africa',
    'Haiti': 'North America', 'Honduras': 'North America', 'Hong Kong': 'Asia', 'Hungary': 'Europe',
    'Iceland': 'Europe', 'India': 'Asia', 'Indonesia': 'Asia', 'Iran': 'Asia',
    'Iraq': 'Asia', 'Ireland': 'Europe', 'Israel': 'Asia', 'Italy': 'Europe',
    'Ivory Coast': 'Africa', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Kazakhstan': 'Asia', 'Kenya': 'Africa', 'Kosovo': 'Europe', 'Kyrgyzstan': 'Asia',
    'Laos': 'Asia', 'Latvia': 'Europe', 'Lebanon': 'Asia', 'Liberia': 'Africa', 'Libya': 'Africa',
    'Lithuania': 'Europe', 'Luxembourg': 'Europe', 'Macedonia': 'Europe', 'Madagascar': 'Africa',
    'Malawi': 'Africa', 'Malaysia': 'Asia', 'Mali': 'Africa', 'Malta': 'Europe',
    'Mauritania': 'Africa', 'Mauritius': 'Africa', 'Mexico': 'North America', 'Moldova': 'Europe',
    'Mongolia': 'Asia', 'Montenegro': 'Europe', 'Morocco': 'Africa', 'Mozambique': 'Africa',
    'Myanmar': 'Asia', 'Namibia': 'Africa', 'Nepal': 'Asia', 'Netherlands': 'Europe',
    'New Zealand': 'Oceania', 'Nicaragua': 'North America', 'Niger': 'Africa', 'Nigeria': 'Africa',
    'North Macedonia': 'Europe', 'Norway': 'Europe', 'Pakistan': 'Asia', 'Palestinian Territories': 'Asia',
    'Panama': 'North America', 'Paraguay': 'South America', 'Peru': 'South America', 'Philippines': 'Asia',
    'Poland': 'Europe', 'Portugal': 'Europe', 'Romania': 'Europe', 'Russia': 'Europe', 'Rwanda': 'Africa',
    'Saudi Arabia': 'Asia', 'Senegal': 'Africa', 'Serbia': 'Europe', 'Sierra Leone': 'Africa',
    'Singapore': 'Asia', 'Slovakia': 'Europe', 'Slovenia': 'Europe', 'South Africa': 'Africa',
    'South Korea': 'Asia', 'Spain': 'Europe', 'Sri Lanka': 'Asia', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Taiwan': 'Asia', 'Tajikistan': 'Asia', 'Tanzania': 'Africa',
    'Thailand': 'Asia', 'Togo': 'Africa', 'Tunisia': 'Africa', 'Turkey': 'Asia',
    'Turkmenistan': 'Asia', 'Uganda': 'Africa', 'Ukraine': 'Europe', 'United Arab Emirates': 'Asia',
    'United Kingdom': 'Europe', 'United States': 'North America', 'Uruguay': 'South America',
    'Uzbekistan': 'Asia', 'Venezuela': 'South America', 'Vietnam': 'Asia', 'Yemen': 'Asia',
    'Zambia': 'Africa', 'Zimbabwe': 'Africa'
}
df['continent'] = df['countrynew'].map(continent_mapping)

# Load ground truth
try:
    gt_df = pd.read_csv("data_final.csv")
    feature_cols = ['countrynew', 'mean_age', 'mean_edu', 'mean_religion',
                   'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth',
                   'hdi_2021', 'mean_temp_2010_2019', 'mean_own_willingness',
                   'mean_other_willingness']
    available_cols = [col for col in feature_cols if col in gt_df.columns]
    gt_df = gt_df[available_cols]
    df = df.merge(gt_df, on='countrynew', how='left')
    df['ground_truth_pi'] = df['mean_other_willingness'] * 100
    
    print(f"✓ Loaded {len(df)} predictions")
    print(f"✓ Countries: {df['countrynew'].nunique()}")
    print(f"✓ Ground truth available: {df['ground_truth_pi'].notna().sum()} observations")
    
except Exception as e:
    print(f"⚠️  Error loading data: {e}")
    exit()

# Prepare data for ML models
country_df = df.groupby('countrynew').first().reset_index()
traditional_features = ['mean_age', 'mean_edu', 'mean_religion',
                       'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth',
                       'hdi_2021', 'mean_temp_2010_2019']
available_features = [f for f in traditional_features if f in country_df.columns
                     and country_df[f].notna().sum() > 0]

# Use Stage 8 for LLM predictions (most information)
stage8_df = df[df['stage'] == 8].copy()
stage8_country = stage8_df.groupby('countrynew').first().reset_index()

# Merge for complete dataset
analysis_df = country_df[['countrynew', 'ground_truth_pi', 'continent'] + available_features].copy()
for model in ['pred_llama', 'pred_gpt', 'pred_claude', 'pred_gemini', 'pred_ensemble']:
    if model in stage8_country.columns:
        analysis_df = analysis_df.merge(
            stage8_country[['countrynew', model]],
            on='countrynew', how='left'
        )

# Remove rows with missing data
analysis_df = analysis_df.dropna()
print(f"✓ Complete cases for analysis: {len(analysis_df)}")

# ================================================================
# 2. Bootstrap Confidence Intervals
# ================================================================

print("\n" + "="*80)
print("2. BOOTSTRAP CONFIDENCE INTERVALS")
print("="*80)
print("Testing if confidence intervals overlap (1000 bootstrap samples)")

def bootstrap_mae(y_true, y_pred, n_bootstrap=1000, confidence=0.95):
    """Calculate bootstrap confidence interval for MAE"""
    maes = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        indices = np.random.choice(n, n, replace=True)
        mae = mean_absolute_error(y_true.iloc[indices], y_pred.iloc[indices])
        maes.append(mae)
    
    alpha = (1 - confidence) / 2
    lower = np.percentile(maes, alpha * 100)
    upper = np.percentile(maes, (1 - alpha) * 100)
    return np.mean(maes), lower, upper

models_to_test = {
    'Llama': 'pred_llama',
    'GPT': 'pred_gpt',
    'Claude': 'pred_claude',
    'Gemini': 'pred_gemini',
    'Ensemble': 'pred_ensemble'
}

bootstrap_results = {}
for model_name, col in models_to_test.items():
    if col in analysis_df.columns:
        mean_mae, lower, upper = bootstrap_mae(
            analysis_df['ground_truth_pi'],
            analysis_df[col]
        )
        bootstrap_results[model_name] = {
            'mean': mean_mae,
            'lower': lower,
            'upper': upper,
            'width': upper - lower
        }
        print(f"\n{model_name}:")
        print(f"   MAE: {mean_mae:.2f}pp")
        print(f"   95% CI: [{lower:.2f}, {upper:.2f}]")
        print(f"   CI Width: {upper - lower:.2f}pp")

# Check for overlapping CIs
print("\n" + "-"*80)
print("Confidence Interval Overlaps:")
if 'Llama' in bootstrap_results:
    llama_ci = bootstrap_results['Llama']
    for model_name, results in bootstrap_results.items():
        if model_name != 'Llama':
            overlap = not (results['lower'] > llama_ci['upper'] or 
                          results['upper'] < llama_ci['lower'])
            if overlap:
                print(f"   Llama vs {model_name}: OVERLAPPING (not significantly different)")
            else:
                if results['mean'] > llama_ci['mean']:
                    print(f"   Llama vs {model_name}: NO OVERLAP (Llama significantly better)")
                else:
                    print(f"   Llama vs {model_name}: NO OVERLAP ({model_name} significantly better)")

# ================================================================
# 3. Permutation Tests
# ================================================================

print("\n" + "="*80)
print("3. PERMUTATION TESTS")
print("="*80)
print("Testing if performance differences could occur by chance (1000 permutations)")

def permutation_test(y_true, y_pred1, y_pred2, n_permutations=1000):
    """
    Test if difference in MAE between two models is significant
    H0: The two models have the same performance
    """
    # Observed difference
    mae1 = mean_absolute_error(y_true, y_pred1)
    mae2 = mean_absolute_error(y_true, y_pred2)
    observed_diff = mae1 - mae2
    
    # Permutation distribution
    perm_diffs = []
    for _ in range(n_permutations):
        # Randomly swap predictions between models
        mask = np.random.random(len(y_true)) > 0.5
        perm_pred1 = np.where(mask, y_pred1, y_pred2)
        perm_pred2 = np.where(mask, y_pred2, y_pred1)
        
        perm_mae1 = mean_absolute_error(y_true, perm_pred1)
        perm_mae2 = mean_absolute_error(y_true, perm_pred2)
        perm_diffs.append(perm_mae1 - perm_mae2)
    
    # P-value: proportion of permutations with difference as extreme as observed
    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))
    
    return observed_diff, p_value

if 'pred_llama' in analysis_df.columns:
    print("\nLlama vs other LLMs:")
    for model_name, col in models_to_test.items():
        if model_name != 'Llama' and col in analysis_df.columns:
            diff, p_val = permutation_test(
                analysis_df['ground_truth_pi'].values,
                analysis_df['pred_llama'].values,
                analysis_df[col].values
            )
            significance = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"   Llama vs {model_name}:")
            print(f"      MAE difference: {diff:.2f}pp")
            print(f"      p-value: {p_val:.4f} {significance}")

# ================================================================
# 4. K-Fold Cross-Validation Stability
# ================================================================

print("\n" + "="*80)
print("4. K-FOLD CROSS-VALIDATION STABILITY")
print("="*80)
print("Testing if Llama consistently outperforms across different data splits (10-fold CV)")

def kfold_evaluation(X, y, llm_preds, n_splits=10, random_state=42):
    """Evaluate models using k-fold cross-validation"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    results = {
        'fold': [],
        'llama_mae': [],
        'ols_mae': [],
        'lasso_mae': []
    }
    
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X)):
        # Split data
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        llm_test = llm_preds.iloc[test_idx]
        
        # Standardize features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train and evaluate OLS
        ols = LinearRegression()
        ols.fit(X_train_scaled, y_train)
        ols_pred = ols.predict(X_test_scaled)
        ols_mae = mean_absolute_error(y_test, ols_pred)
        
        # Train and evaluate Lasso
        lasso = LassoCV(cv=5, random_state=42)
        lasso.fit(X_train_scaled, y_train)
        lasso_pred = lasso.predict(X_test_scaled)
        lasso_mae = mean_absolute_error(y_test, lasso_pred)
        
        # Evaluate Llama (no training needed)
        llama_mae = mean_absolute_error(y_test, llm_test)
        
        results['fold'].append(fold_idx + 1)
        results['llama_mae'].append(llama_mae)
        results['ols_mae'].append(ols_mae)
        results['lasso_mae'].append(lasso_mae)
    
    return pd.DataFrame(results)

if 'pred_llama' in analysis_df.columns and len(available_features) > 0:
    X = analysis_df[available_features]
    y = analysis_df['ground_truth_pi']
    llm_preds = analysis_df['pred_llama']
    
    kfold_df = kfold_evaluation(X, y, llm_preds)
    
    print("\nMean MAE across 10 folds:")
    print(f"   Llama:  {kfold_df['llama_mae'].mean():.2f} ± {kfold_df['llama_mae'].std():.2f}pp")
    print(f"   OLS:    {kfold_df['ols_mae'].mean():.2f} ± {kfold_df['ols_mae'].std():.2f}pp")
    print(f"   Lasso:  {kfold_df['lasso_mae'].mean():.2f} ± {kfold_df['lasso_mae'].std():.2f}pp")
    
    # Count how many folds Llama wins
    llama_wins = (kfold_df['llama_mae'] < kfold_df['ols_mae']).sum()
    llama_wins_lasso = (kfold_df['llama_mae'] < kfold_df['lasso_mae']).sum()
    
    print(f"\nLlama wins vs OLS: {llama_wins}/10 folds ({llama_wins*10}%)")
    print(f"Llama wins vs Lasso: {llama_wins_lasso}/10 folds ({llama_wins_lasso*10}%)")
    
    # Statistical test
    t_stat, p_val = ttest_rel(kfold_df['llama_mae'], kfold_df['ols_mae'])
    print(f"\nPaired t-test (Llama vs OLS): t = {t_stat:.3f}, p = {p_val:.4f}")
    
    t_stat_lasso, p_val_lasso = ttest_rel(kfold_df['llama_mae'], kfold_df['lasso_mae'])
    print(f"Paired t-test (Llama vs Lasso): t = {t_stat_lasso:.3f}, p = {p_val_lasso:.4f}")
    
    # Save results
    kfold_df.to_csv('llama_kfold_stability.csv', index=False)
    print("\n✓ Saved llama_kfold_stability.csv")

# ================================================================
# 5. Subgroup Analysis
# ================================================================

print("\n" + "="*80)
print("5. SUBGROUP ROBUSTNESS ANALYSIS")
print("="*80)
print("Testing if Llama performs consistently across different subgroups")

if 'pred_llama' in analysis_df.columns and 'continent' in analysis_df.columns:
    print("\n5.1 Performance by Continent:")
    print("-" * 60)
    
    continent_results = []
    for continent in sorted(analysis_df['continent'].dropna().unique()):
        subset = analysis_df[analysis_df['continent'] == continent]
        if len(subset) >= 5:  # Only analyze if enough samples
            llama_mae = mean_absolute_error(subset['ground_truth_pi'], subset['pred_llama'])
            
            # Compare to other models
            comparisons = {}
            for model_name, col in models_to_test.items():
                if model_name != 'Llama' and col in subset.columns:
                    model_mae = mean_absolute_error(subset['ground_truth_pi'], subset[col])
                    comparisons[model_name] = model_mae
            
            print(f"\n{continent} (n={len(subset)}):")
            print(f"   Llama MAE: {llama_mae:.2f}pp")
            for model_name, mae in comparisons.items():
                diff = mae - llama_mae
                status = "✓ Llama better" if diff > 0 else "✗ Llama worse"
                print(f"   {model_name} MAE: {mae:.2f}pp (diff: {diff:+.2f}pp) {status}")
            
            continent_results.append({
                'continent': continent,
                'n': len(subset),
                'llama_mae': llama_mae
            })
    
    continent_df = pd.DataFrame(continent_results)
    continent_df.to_csv('llama_continent_performance.csv', index=False)
    print("\n✓ Saved llama_continent_performance.csv")

# Quartile analysis
if 'pred_llama' in analysis_df.columns and 'gdp_capita_2021' in analysis_df.columns:
    print("\n5.2 Performance by GDP Quartile:")
    print("-" * 60)
    
    analysis_df['gdp_quartile'] = pd.qcut(analysis_df['gdp_capita_2021'], 
                                           q=4, labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'])
    
    for quartile in ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']:
        subset = analysis_df[analysis_df['gdp_quartile'] == quartile]
        if len(subset) > 0:
            llama_mae = mean_absolute_error(subset['ground_truth_pi'], subset['pred_llama'])
            print(f"\n{quartile} (n={len(subset)}):")
            print(f"   Llama MAE: {llama_mae:.2f}pp")

# ================================================================
# 6. Error Pattern Analysis
# ================================================================

print("\n" + "="*80)
print("6. ERROR PATTERN ANALYSIS")
print("="*80)
print("Analyzing if Llama has systematic biases or just random errors")

if 'pred_llama' in analysis_df.columns:
    # Calculate errors
    analysis_df['llama_error'] = analysis_df['pred_llama'] - analysis_df['ground_truth_pi']
    analysis_df['llama_abs_error'] = np.abs(analysis_df['llama_error'])
    
    print("\n6.1 Error Distribution:")
    print(f"   Mean Error (bias): {analysis_df['llama_error'].mean():.2f}pp")
    print(f"   Median Error: {analysis_df['llama_error'].median():.2f}pp")
    print(f"   Error Std Dev: {analysis_df['llama_error'].std():.2f}pp")
    print(f"   Skewness: {stats.skew(analysis_df['llama_error']):.3f}")
    print(f"   Kurtosis: {stats.kurtosis(analysis_df['llama_error']):.3f}")
    
    # Test for systematic bias
    t_stat, p_val = stats.ttest_1samp(analysis_df['llama_error'], 0)
    print(f"\n6.2 Systematic Bias Test:")
    print(f"   H0: Mean error = 0 (no systematic bias)")
    print(f"   t = {t_stat:.3f}, p = {p_val:.4f}")
    if p_val < 0.05:
        print(f"   ⚠️  Significant systematic bias detected")
    else:
        print(f"   ✓ No significant systematic bias")
    
    # Identify outliers
    print(f"\n6.3 Outlier Analysis:")
    outlier_threshold = analysis_df['llama_abs_error'].quantile(0.90)
    outliers = analysis_df[analysis_df['llama_abs_error'] > outlier_threshold]
    print(f"   Countries with largest errors (top 10%):")
    for _, row in outliers.nsmallest(10, 'llama_abs_error').iterrows():
        print(f"      {row['countrynew']}: {row['llama_error']:.1f}pp error")

# ================================================================
# 7. Multiple Random Seeds Test
# ================================================================

print("\n" + "="*80)
print("7. CONSISTENCY ACROSS RANDOM SEEDS")
print("="*80)
print("Testing if results hold with different random initializations (20 seeds)")

def evaluate_with_seed(X, y, llm_preds, seed):
    """Evaluate models with a specific random seed"""
    # 80-20 split - need to do this in two steps
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    
    # Split llm_preds using the same indices
    _, llm_test = train_test_split(
        llm_preds, test_size=0.2, random_state=seed
    )
    
    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # OLS
    ols = LinearRegression()
    ols.fit(X_train_scaled, y_train)
    ols_mae = mean_absolute_error(y_test, ols.predict(X_test_scaled))
    
    # Lasso
    lasso = LassoCV(cv=5, random_state=seed)
    lasso.fit(X_train_scaled, y_train)
    lasso_mae = mean_absolute_error(y_test, lasso.predict(X_test_scaled))
    
    # Llama
    llama_mae = mean_absolute_error(y_test, llm_test)
    
    return llama_mae, ols_mae, lasso_mae

if 'pred_llama' in analysis_df.columns and len(available_features) > 0:
    seeds = range(42, 62)  # 20 different seeds
    seed_results = []
    
    X = analysis_df[available_features]
    y = analysis_df['ground_truth_pi']
    llm_preds = analysis_df['pred_llama']
    
    for seed in seeds:
        llama_mae, ols_mae, lasso_mae = evaluate_with_seed(X, y, llm_preds, seed)
        seed_results.append({
            'seed': seed,
            'llama_mae': llama_mae,
            'ols_mae': ols_mae,
            'lasso_mae': lasso_mae,
            'llama_wins_ols': llama_mae < ols_mae,
            'llama_wins_lasso': llama_mae < lasso_mae
        })
    
    seed_df = pd.DataFrame(seed_results)
    
    print(f"\nResults across 20 random seeds:")
    print(f"   Llama MAE: {seed_df['llama_mae'].mean():.2f} ± {seed_df['llama_mae'].std():.2f}pp")
    print(f"   OLS MAE:   {seed_df['ols_mae'].mean():.2f} ± {seed_df['ols_mae'].std():.2f}pp")
    print(f"   Lasso MAE: {seed_df['lasso_mae'].mean():.2f} ± {seed_df['lasso_mae'].std():.2f}pp")
    
    print(f"\nConsistency:")
    llama_win_rate_ols = seed_df['llama_wins_ols'].mean() * 100
    llama_win_rate_lasso = seed_df['llama_wins_lasso'].mean() * 100
    print(f"   Llama beats OLS: {llama_win_rate_ols:.0f}% of seeds")
    print(f"   Llama beats Lasso: {llama_win_rate_lasso:.0f}% of seeds")
    
    if llama_win_rate_ols >= 50:
        print(f"   ✓ Llama consistently competitive with OLS")
    else:
        print(f"   ✗ Llama inconsistently competitive with OLS")
    
    # Save results
    seed_df.to_csv('llama_seed_consistency.csv', index=False)
    print("\n✓ Saved llama_seed_consistency.csv")

# ================================================================
# 8. Create Summary Visualization
# ================================================================

print("\n" + "="*80)
print("8. CREATING VISUALIZATIONS")
print("="*80)

fig = plt.figure()
fig.set_size_inches(18, 12)
fig.set_dpi(300)
matplotlib.rcParams.update({})
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Bootstrap CIs
if bootstrap_results:
    ax1 = fig.add_subplot(gs[0, :2])
    models = list(bootstrap_results.keys())
    means = [bootstrap_results[m]['mean'] for m in models]
    lowers = [bootstrap_results[m]['lower'] for m in models]
    uppers = [bootstrap_results[m]['upper'] for m in models]
    
    ax1.errorbar(models, means, 
                yerr=[np.array(means) - np.array(lowers), 
                      np.array(uppers) - np.array(means)],
                fmt='o', capsize=5, capthick=2, markersize=8)
    ax1.axhline(y=bootstrap_results['Llama']['mean'], 
               color='red', linestyle='--', alpha=0.5, label='Llama mean')
    ax1.set_ylabel('Test MAE (pp)', fontsize=12)
    ax1.set_title('Bootstrap 95% Confidence Intervals', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

# 2. K-fold stability
if 'kfold_df' in locals():
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.boxplot([kfold_df['llama_mae'], kfold_df['ols_mae'], kfold_df['lasso_mae']],
                labels=['Llama', 'OLS', 'Lasso'])
    ax2.set_ylabel('Test MAE (pp)', fontsize=12)
    ax2.set_title('10-Fold CV Stability', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)

# 3. Error distribution
if 'llama_error' in analysis_df.columns:
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.hist(analysis_df['llama_error'], bins=30, edgecolor='black', alpha=0.7)
    ax3.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No bias')
    ax3.axvline(x=analysis_df['llama_error'].mean(), 
               color='blue', linestyle='-', linewidth=2, label=f'Mean: {analysis_df["llama_error"].mean():.1f}pp')
    ax3.set_xlabel('Prediction Error (pp)', fontsize=12)
    ax3.set_ylabel('Frequency', fontsize=12)
    ax3.set_title('Llama Error Distribution', fontsize=14, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

# 4. Prediction vs actual
if 'pred_llama' in analysis_df.columns:
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.scatter(analysis_df['ground_truth_pi'], analysis_df['pred_llama'], alpha=0.6)
    ax4.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect prediction')
    ax4.set_xlabel('Ground Truth PI (%)', fontsize=12)
    ax4.set_ylabel('Llama Prediction (%)', fontsize=12)
    ax4.set_title('Llama: Predicted vs Actual', fontsize=14, fontweight='bold')
    
    # Add correlation
    r, p = pearsonr(analysis_df['ground_truth_pi'], analysis_df['pred_llama'])
    ax4.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.4f}', 
            transform=ax4.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax4.legend()
    ax4.grid(True, alpha=0.3)

# 5. Residuals vs predicted
if 'llama_error' in analysis_df.columns:
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.scatter(analysis_df['pred_llama'], analysis_df['llama_error'], alpha=0.6)
    ax5.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax5.set_xlabel('Predicted PI (%)', fontsize=12)
    ax5.set_ylabel('Residual (pp)', fontsize=12)
    ax5.set_title('Residual Plot', fontsize=14, fontweight='bold')
    ax5.grid(True, alpha=0.3)

# 6. Continent performance
if 'continent_df' in locals():
    ax6 = fig.add_subplot(gs[2, 0])
    continent_df = continent_df.sort_values('llama_mae')
    ax6.barh(continent_df['continent'], continent_df['llama_mae'])
    ax6.set_xlabel('MAE (pp)', fontsize=12)
    ax6.set_title('Llama MAE by Continent', fontsize=14, fontweight='bold')
    ax6.grid(True, alpha=0.3, axis='x')

# 7. Seed consistency
if 'seed_df' in locals():
    ax7 = fig.add_subplot(gs[2, 1])
    ax7.scatter(seed_df['seed'], seed_df['llama_mae'], label='Llama', alpha=0.7)
    ax7.scatter(seed_df['seed'], seed_df['ols_mae'], label='OLS', alpha=0.7)
    ax7.scatter(seed_df['seed'], seed_df['lasso_mae'], label='Lasso', alpha=0.7)
    ax7.set_xlabel('Random Seed', fontsize=12)
    ax7.set_ylabel('Test MAE (pp)', fontsize=12)
    ax7.set_title('Consistency Across Seeds', fontsize=14, fontweight='bold')
    ax7.legend()
    ax7.grid(True, alpha=0.3)

# 8. Model ranking frequency
if 'seed_df' in locals():
    ax8 = fig.add_subplot(gs[2, 2])
    rankings = []
    for _, row in seed_df.iterrows():
        maes = [('Llama', row['llama_mae']), ('OLS', row['ols_mae']), ('Lasso', row['lasso_mae'])]
        sorted_models = sorted(maes, key=lambda x: x[1])
        rankings.append(sorted_models[0][0])
    
    from collections import Counter
    rank_counts = Counter(rankings)
    models_ranked = list(rank_counts.keys())
    counts = [rank_counts[m] for m in models_ranked]
    
    ax8.bar(models_ranked, counts)
    ax8.set_ylabel('Frequency (out of 20)', fontsize=12)
    ax8.set_title('Best Model Frequency', fontsize=14, fontweight='bold')
    ax8.grid(True, alpha=0.3, axis='y')

plt.suptitle('LLAMA ROBUSTNESS ANALYSIS', fontsize=16, fontweight='bold', y=0.995)
plt.savefig('llama_robustness_comprehensive.png', dpi=300, bbox_inches='tight')
plt.savefig('llama_robustness_comprehensive.pdf', bbox_inches='tight')
print("✓ Saved llama_robustness_comprehensive.png")
print("✓ Saved llama_robustness_comprehensive.pdf")
plt.close()

# ================================================================
# 9. Final Summary Report
# ================================================================

print("\n" + "="*80)
print("9. GENERATING SUMMARY REPORT")
print("="*80)

with open('llama_robustness_report.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("LLAMA PERFORMANCE VALIDATION REPORT\n")
    f.write("="*80 + "\n\n")
    
    f.write("QUESTION: Is Llama's superior performance genuine or a statistical fluke?\n\n")
    
    f.write("="*80 + "\n")
    f.write("1. BOOTSTRAP CONFIDENCE INTERVALS\n")
    f.write("="*80 + "\n")
    if bootstrap_results:
        for model, results in bootstrap_results.items():
            f.write(f"\n{model}:\n")
            f.write(f"   MAE: {results['mean']:.2f}pp\n")
            f.write(f"   95% CI: [{results['lower']:.2f}, {results['upper']:.2f}]\n")
        
        if 'Llama' in bootstrap_results:
            f.write("\nCI Overlap Analysis:\n")
            llama_ci = bootstrap_results['Llama']
            for model_name, results in bootstrap_results.items():
                if model_name != 'Llama':
                    overlap = not (results['lower'] > llama_ci['upper'] or 
                                  results['upper'] < llama_ci['lower'])
                    f.write(f"   Llama vs {model_name}: {'OVERLAPPING' if overlap else 'NO OVERLAP'}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("2. K-FOLD CROSS-VALIDATION\n")
    f.write("="*80 + "\n")
    if 'kfold_df' in locals():
        f.write(f"\nMean MAE across 10 folds:\n")
        f.write(f"   Llama:  {kfold_df['llama_mae'].mean():.2f} ± {kfold_df['llama_mae'].std():.2f}pp\n")
        f.write(f"   OLS:    {kfold_df['ols_mae'].mean():.2f} ± {kfold_df['ols_mae'].std():.2f}pp\n")
        f.write(f"   Lasso:  {kfold_df['lasso_mae'].mean():.2f} ± {kfold_df['lasso_mae'].std():.2f}pp\n")
        
        llama_wins = (kfold_df['llama_mae'] < kfold_df['ols_mae']).sum()
        llama_wins_lasso = (kfold_df['llama_mae'] < kfold_df['lasso_mae']).sum()
        f.write(f"\nLlama wins vs OLS: {llama_wins}/10 folds\n")
        f.write(f"Llama wins vs Lasso: {llama_wins_lasso}/10 folds\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("3. RANDOM SEED CONSISTENCY\n")
    f.write("="*80 + "\n")
    if 'seed_df' in locals():
        f.write(f"\nResults across 20 random seeds:\n")
        f.write(f"   Llama MAE: {seed_df['llama_mae'].mean():.2f} ± {seed_df['llama_mae'].std():.2f}pp\n")
        f.write(f"   OLS MAE:   {seed_df['ols_mae'].mean():.2f} ± {seed_df['ols_mae'].std():.2f}pp\n")
        f.write(f"   Lasso MAE: {seed_df['lasso_mae'].mean():.2f} ± {seed_df['lasso_mae'].std():.2f}pp\n")
        
        llama_win_rate_ols = seed_df['llama_wins_ols'].mean() * 100
        llama_win_rate_lasso = seed_df['llama_wins_lasso'].mean() * 100
        f.write(f"\n   Llama beats OLS: {llama_win_rate_ols:.0f}% of seeds\n")
        f.write(f"   Llama beats Lasso: {llama_win_rate_lasso:.0f}% of seeds\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("4. ERROR PATTERN ANALYSIS\n")
    f.write("="*80 + "\n")
    if 'llama_error' in analysis_df.columns:
        f.write(f"\nError Statistics:\n")
        f.write(f"   Mean Error (bias): {analysis_df['llama_error'].mean():.2f}pp\n")
        f.write(f"   Std Dev: {analysis_df['llama_error'].std():.2f}pp\n")
        f.write(f"   Skewness: {stats.skew(analysis_df['llama_error']):.3f}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("CONCLUSION\n")
    f.write("="*80 + "\n")
    f.write("\nBased on comprehensive robustness testing:\n\n")
    
    # Automated conclusion
    conclusion_points = []
    
    if 'kfold_df' in locals():
        llama_wins = (kfold_df['llama_mae'] < kfold_df['ols_mae']).sum()
        if llama_wins >= 5:
            conclusion_points.append("✓ Llama performance is CONSISTENT across k-fold validation")
        else:
            conclusion_points.append("✗ Llama performance is INCONSISTENT across k-fold validation")
    
    if 'seed_df' in locals():
        llama_win_rate = seed_df['llama_wins_ols'].mean() * 100
        if llama_win_rate >= 50:
            conclusion_points.append("✓ Llama performance is ROBUST across random seeds")
        else:
            conclusion_points.append("✗ Llama performance is SENSITIVE to random initialization")
    
    if bootstrap_results and 'Llama' in bootstrap_results:
        llama_ci = bootstrap_results['Llama']
        ols_better = False
        if 'OLS' in bootstrap_results:
            ols_ci = bootstrap_results['OLS']
            if ols_ci['upper'] < llama_ci['lower']:
                ols_better = True
        
        if not ols_better:
            conclusion_points.append("✓ Llama performance is STATISTICALLY COMPETITIVE")
        else:
            conclusion_points.append("✗ OLS significantly outperforms Llama")
    
    for point in conclusion_points:
        f.write(point + "\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("FILES GENERATED:\n")
    f.write("   - llama_robustness_report.txt (this file)\n")
    f.write("   - llama_robustness_comprehensive.png (visualization)\n")
    f.write("   - llama_robustness_comprehensive.pdf (visualization)\n")
    f.write("   - llama_kfold_stability.csv\n")
    f.write("   - llama_seed_consistency.csv\n")
    f.write("   - llama_continent_performance.csv\n")

print("✓ Saved llama_robustness_report.txt")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nKey files created:")
print("   1. llama_robustness_report.txt - Comprehensive text summary")
print("   2. llama_robustness_comprehensive.png - 8-panel visualization")
print("   3. llama_robustness_comprehensive.pdf - 8-panel visualization (PDF)")
print("   4. llama_kfold_stability.csv - K-fold CV results")
print("   5. llama_seed_consistency.csv - Random seed test results")
print("   6. llama_continent_performance.csv - Subgroup analysis")
print("\n" + "="*80)

LLAMA PERFORMANCE VALIDATION & ROBUSTNESS TESTING

1. LOADING DATA
✓ Loaded 1000 predictions
✓ Countries: 125
✓ Ground truth available: 1000 observations
✓ Complete cases for analysis: 114

2. BOOTSTRAP CONFIDENCE INTERVALS
Testing if confidence intervals overlap (1000 bootstrap samples)

Llama:
   MAE: 6.80pp
   95% CI: [5.95, 7.71]
   CI Width: 1.76pp

GPT:
   MAE: 13.62pp
   95% CI: [12.48, 14.74]
   CI Width: 2.26pp

Claude:
   MAE: 4.89pp
   95% CI: [4.16, 5.72]
   CI Width: 1.56pp

Gemini:
   MAE: 14.96pp
   95% CI: [13.49, 16.52]
   CI Width: 3.04pp

Ensemble:
   MAE: 9.09pp
   95% CI: [8.24, 9.94]
   CI Width: 1.70pp

--------------------------------------------------------------------------------
Confidence Interval Overlaps:
   Llama vs GPT: NO OVERLAP (Llama significantly better)
   Llama vs Claude: NO OVERLAP (Claude significantly better)
   Llama vs Gemini: NO OVERLAP (Llama significantly better)
   Llama vs Ensemble: NO OVERLAP (Llama significantly better)

3. PERMUTATION

In [2]:
#!/usr/bin/env python3
"""
CLAUDE ROBUSTNESS TESTING
========================
Comprehensive statistical tests to determine if Claude's superior performance
is genuine or a statistical fluke.

Tests included:
1. Statistical significance testing (bootstrap & permutation tests)
2. Cross-validation stability analysis
3. Robustness checks across subgroups
4. Error pattern analysis
5. Consistency checks with different random seeds
6. Performance degradation tests
"""

# ================================================================
# Fix matplotlib compatibility
# ================================================================
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, ttest_rel, wilcoxon
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.preprocessing import StandardScaler

# Set style
sns.set_style("whitegrid")

print("="*80)
print("CLAUDE PERFORMANCE VALIDATION & ROBUSTNESS TESTING")
print("="*80)

# ================================================================
# 1. Load Data
# ================================================================

print("\n" + "="*80)
print("1. LOADING DATA")
print("="*80)

df = pd.read_csv("predictions_all_stages_long.csv")
model_cols = [col for col in df.columns if col.startswith('pred_')]
df['pred_ensemble'] = df[model_cols].mean(axis=1)

# Add continent mapping
continent_mapping = {
    'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'Argentina': 'South America',
    'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europe', 'Bangladesh': 'Asia',
    'Belgium': 'Europe', 'Benin': 'Africa', 'Bolivia': 'South America', 'Bosnia Herzegovina': 'Europe',
    'Botswana': 'Africa', 'Brazil': 'South America', 'Bulgaria': 'Europe', 'Burkina Faso': 'Africa',
    'Cambodia': 'Asia', 'Cameroon': 'Africa', 'Canada': 'North America', 'Chad': 'Africa',
    'Chile': 'South America', 'China': 'Asia', 'Colombia': 'South America', 'Congo Brazzaville': 'Africa',
    'Costa Rica': 'North America', 'Croatia': 'Europe', 'Cyprus': 'Europe', 'Czech Republic': 'Europe',
    'Denmark': 'Europe', 'Dominican Republic': 'North America', 'Ecuador': 'South America', 'Egypt': 'Africa',
    'El Salvador': 'North America', 'Estonia': 'Europe', 'Ethiopia': 'Africa', 'Finland': 'Europe',
    'France': 'Europe', 'Gabon': 'Africa', 'Georgia': 'Asia', 'Germany': 'Europe',
    'Ghana': 'Africa', 'Greece': 'Europe', 'Guatemala': 'North America', 'Guinea': 'Africa',
    'Haiti': 'North America', 'Honduras': 'North America', 'Hong Kong': 'Asia', 'Hungary': 'Europe',
    'Iceland': 'Europe', 'India': 'Asia', 'Indonesia': 'Asia', 'Iran': 'Asia',
    'Iraq': 'Asia', 'Ireland': 'Europe', 'Israel': 'Asia', 'Italy': 'Europe',
    'Ivory Coast': 'Africa', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Kazakhstan': 'Asia', 'Kenya': 'Africa', 'Kosovo': 'Europe', 'Kyrgyzstan': 'Asia',
    'Laos': 'Asia', 'Latvia': 'Europe', 'Lebanon': 'Asia', 'Liberia': 'Africa', 'Libya': 'Africa',
    'Lithuania': 'Europe', 'Luxembourg': 'Europe', 'Macedonia': 'Europe', 'Madagascar': 'Africa',
    'Malawi': 'Africa', 'Malaysia': 'Asia', 'Mali': 'Africa', 'Malta': 'Europe',
    'Mauritania': 'Africa', 'Mauritius': 'Africa', 'Mexico': 'North America', 'Moldova': 'Europe',
    'Mongolia': 'Asia', 'Montenegro': 'Europe', 'Morocco': 'Africa', 'Mozambique': 'Africa',
    'Myanmar': 'Asia', 'Namibia': 'Africa', 'Nepal': 'Asia', 'Netherlands': 'Europe',
    'New Zealand': 'Oceania', 'Nicaragua': 'North America', 'Niger': 'Africa', 'Nigeria': 'Africa',
    'North Macedonia': 'Europe', 'Norway': 'Europe', 'Pakistan': 'Asia', 'Palestinian Territories': 'Asia',
    'Panama': 'North America', 'Paraguay': 'South America', 'Peru': 'South America', 'Philippines': 'Asia',
    'Poland': 'Europe', 'Portugal': 'Europe', 'Romania': 'Europe', 'Russia': 'Europe', 'Rwanda': 'Africa',
    'Saudi Arabia': 'Asia', 'Senegal': 'Africa', 'Serbia': 'Europe', 'Sierra Leone': 'Africa',
    'Singapore': 'Asia', 'Slovakia': 'Europe', 'Slovenia': 'Europe', 'South Africa': 'Africa',
    'South Korea': 'Asia', 'Spain': 'Europe', 'Sri Lanka': 'Asia', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Taiwan': 'Asia', 'Tajikistan': 'Asia', 'Tanzania': 'Africa',
    'Thailand': 'Asia', 'Togo': 'Africa', 'Tunisia': 'Africa', 'Turkey': 'Asia',
    'Turkmenistan': 'Asia', 'Uganda': 'Africa', 'Ukraine': 'Europe', 'United Arab Emirates': 'Asia',
    'United Kingdom': 'Europe', 'United States': 'North America', 'Uruguay': 'South America',
    'Uzbekistan': 'Asia', 'Venezuela': 'South America', 'Vietnam': 'Asia', 'Yemen': 'Asia',
    'Zambia': 'Africa', 'Zimbabwe': 'Africa'
}
df['continent'] = df['countrynew'].map(continent_mapping)

# Load ground truth
try:
    gt_df = pd.read_csv("data_final.csv")
    feature_cols = ['countrynew', 'mean_age', 'mean_edu', 'mean_religion',
                   'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth',
                   'hdi_2021', 'mean_temp_2010_2019', 'mean_own_willingness',
                   'mean_other_willingness']
    available_cols = [col for col in feature_cols if col in gt_df.columns]
    gt_df = gt_df[available_cols]
    df = df.merge(gt_df, on='countrynew', how='left')
    df['ground_truth_pi'] = df['mean_other_willingness'] * 100
    
    print(f"✓ Loaded {len(df)} predictions")
    print(f"✓ Countries: {df['countrynew'].nunique()}")
    print(f"✓ Ground truth available: {df['ground_truth_pi'].notna().sum()} observations")
    
except Exception as e:
    print(f"⚠️  Error loading data: {e}")
    exit()

# Prepare data for ML models
country_df = df.groupby('countrynew').first().reset_index()
traditional_features = ['mean_age', 'mean_edu', 'mean_religion',
                       'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth',
                       'hdi_2021', 'mean_temp_2010_2019']
available_features = [f for f in traditional_features if f in country_df.columns
                     and country_df[f].notna().sum() > 0]

# Use Stage 8 for LLM predictions (most information)
stage8_df = df[df['stage'] == 8].copy()
stage8_country = stage8_df.groupby('countrynew').first().reset_index()

# Merge for complete dataset
analysis_df = country_df[['countrynew', 'ground_truth_pi', 'continent'] + available_features].copy()
for model in ['pred_claude', 'pred_gpt', 'pred_llama', 'pred_gemini', 'pred_ensemble']:
    if model in stage8_country.columns:
        analysis_df = analysis_df.merge(
            stage8_country[['countrynew', model]],
            on='countrynew', how='left'
        )

# Remove rows with missing data
print(f"\nBefore dropna: {len(analysis_df)} rows")
print(f"Columns: {analysis_df.columns.tolist()}")
print(f"Missing values per column:")
print(analysis_df.isnull().sum())

analysis_df = analysis_df.dropna()
print(f"\n✓ Complete cases for analysis: {len(analysis_df)}")

if len(analysis_df) == 0:
    print("\n" + "="*80)
    print("❌ ERROR: No complete cases after removing missing data!")
    print("="*80)
    print("This means all rows have at least one missing value.")
    print("Check your data_final.csv for missing values in:")
    print("  • ground_truth_pi")
    print("  • continent") 
    print("  • prediction columns (pred_claude, pred_gpt, etc.)")
    exit()

# ================================================================
# 2. Bootstrap Confidence Intervals
# ================================================================

print("\n" + "="*80)
print("2. BOOTSTRAP CONFIDENCE INTERVALS")
print("="*80)
print("Testing if confidence intervals overlap (1000 bootstrap samples)")

def bootstrap_mae(y_true, y_pred, n_bootstrap=1000, confidence=0.95):
    """Calculate bootstrap confidence interval for MAE"""
    maes = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        indices = np.random.choice(n, n, replace=True)
        mae = mean_absolute_error(y_true.iloc[indices], y_pred.iloc[indices])
        maes.append(mae)
    
    alpha = (1 - confidence) / 2
    lower = np.percentile(maes, alpha * 100)
    upper = np.percentile(maes, (1 - alpha) * 100)
    return np.mean(maes), lower, upper

models_to_test = {
    'Claude': 'pred_claude',
    'GPT': 'pred_gpt',
    'Llama': 'pred_llama',
    'Gemini': 'pred_gemini',
    'Ensemble': 'pred_ensemble'
}

bootstrap_results = {}
for model_name, col in models_to_test.items():
    print(f"Checking {model_name} ({col})...")
    if col in analysis_df.columns:
        try:
            mean_mae, lower, upper = bootstrap_mae(
                analysis_df['ground_truth_pi'],
                analysis_df[col]
            )
            bootstrap_results[model_name] = {
                'mean': mean_mae,
                'lower': lower,
                'upper': upper,
                'width': upper - lower
            }
            print(f"\n{model_name}:")
            print(f"   MAE: {mean_mae:.2f}pp")
            print(f"   95% CI: [{lower:.2f}, {upper:.2f}]")
            print(f"   CI Width: {upper - lower:.2f}pp")
        except Exception as e:
            print(f"   ✗ Error calculating bootstrap for {model_name}: {e}")
    else:
        print(f"   ✗ Column {col} not found in analysis_df")

if len(bootstrap_results) == 0:
    print("\n" + "="*80)
    print("❌ ERROR: No models could be evaluated!")
    print("="*80)
    print("No bootstrap results were generated. This means:")
    print("  • None of the prediction columns were found in analysis_df, OR")
    print("  • All bootstrap calculations failed")
    print(f"\nanalysis_df columns: {analysis_df.columns.tolist()}")
    print(f"\nExpected columns: {list(models_to_test.values())}")
    exit()

# Check for overlapping CIs
print("\n" + "-"*80)
print("Confidence Interval Overlaps:")
if 'Claude' in bootstrap_results:
    claude_ci = bootstrap_results['Claude']
    for model_name, results in bootstrap_results.items():
        if model_name != 'Claude':
            overlap = not (results['lower'] > claude_ci['upper'] or 
                          results['upper'] < claude_ci['lower'])
            if overlap:
                print(f"   Claude vs {model_name}: OVERLAPPING (not significantly different)")
            else:
                if results['mean'] > claude_ci['mean']:
                    print(f"   Claude vs {model_name}: NO OVERLAP (Claude significantly better)")
                else:
                    print(f"   Claude vs {model_name}: NO OVERLAP ({model_name} significantly better)")

# ================================================================
# 3. Permutation Tests
# ================================================================

print("\n" + "="*80)
print("3. PERMUTATION TESTS")
print("="*80)
print("Testing if performance differences could occur by chance (1000 permutations)")

def permutation_test(y_true, y_pred1, y_pred2, n_permutations=1000):
    """
    Test if difference in MAE between two models is significant
    H0: The two models have the same performance
    """
    # Observed difference
    mae1 = mean_absolute_error(y_true, y_pred1)
    mae2 = mean_absolute_error(y_true, y_pred2)
    observed_diff = mae1 - mae2
    
    # Permutation distribution
    perm_diffs = []
    for _ in range(n_permutations):
        # Randomly swap predictions between models
        mask = np.random.random(len(y_true)) > 0.5
        perm_pred1 = np.where(mask, y_pred1, y_pred2)
        perm_pred2 = np.where(mask, y_pred2, y_pred1)
        
        perm_mae1 = mean_absolute_error(y_true, perm_pred1)
        perm_mae2 = mean_absolute_error(y_true, perm_pred2)
        perm_diffs.append(perm_mae1 - perm_mae2)
    
    # P-value: proportion of permutations with difference as extreme as observed
    p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))
    
    return observed_diff, p_value

if 'pred_claude' in analysis_df.columns:
    print("\nClaude vs other LLMs:")
    for model_name, col in models_to_test.items():
        if model_name != 'Claude' and col in analysis_df.columns:
            diff, p_val = permutation_test(
                analysis_df['ground_truth_pi'].values,
                analysis_df['pred_claude'].values,
                analysis_df[col].values
            )
            significance = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            print(f"   Claude vs {model_name}:")
            print(f"      MAE difference: {diff:.2f}pp")
            print(f"      p-value: {p_val:.4f} {significance}")

# ================================================================
# 4. K-Fold Cross-Validation Stability
# ================================================================

print("\n" + "="*80)
print("4. K-FOLD CROSS-VALIDATION STABILITY")
print("="*80)
print("Testing if Claude consistently outperforms across different data splits (10-fold CV)")

def kfold_evaluation(X, y, llm_preds, n_splits=10, random_state=42):
    """Evaluate models using k-fold cross-validation"""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    results = {
        'fold': [],
        'claude_mae': [],
        'ols_mae': [],
        'lasso_mae': []
    }
    
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X)):
        # Split data
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        llm_test = llm_preds.iloc[test_idx]
        
        # Standardize features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train and evaluate OLS
        ols = LinearRegression()
        ols.fit(X_train_scaled, y_train)
        ols_pred = ols.predict(X_test_scaled)
        ols_mae = mean_absolute_error(y_test, ols_pred)
        
        # Train and evaluate Lasso
        lasso = LassoCV(cv=5, random_state=42)
        lasso.fit(X_train_scaled, y_train)
        lasso_pred = lasso.predict(X_test_scaled)
        lasso_mae = mean_absolute_error(y_test, lasso_pred)
        
        # Evaluate Claude (no training needed)
        claude_mae = mean_absolute_error(y_test, llm_test)
        
        results['fold'].append(fold_idx + 1)
        results['claude_mae'].append(claude_mae)
        results['ols_mae'].append(ols_mae)
        results['lasso_mae'].append(lasso_mae)
    
    return pd.DataFrame(results)

if 'pred_claude' in analysis_df.columns and len(available_features) > 0:
    X = analysis_df[available_features]
    y = analysis_df['ground_truth_pi']
    llm_preds = analysis_df['pred_claude']
    
    kfold_df = kfold_evaluation(X, y, llm_preds)
    
    print("\nMean MAE across 10 folds:")
    print(f"   Claude:  {kfold_df['claude_mae'].mean():.2f} ± {kfold_df['claude_mae'].std():.2f}pp")
    print(f"   OLS:    {kfold_df['ols_mae'].mean():.2f} ± {kfold_df['ols_mae'].std():.2f}pp")
    print(f"   Lasso:  {kfold_df['lasso_mae'].mean():.2f} ± {kfold_df['lasso_mae'].std():.2f}pp")
    
    # Count how many folds Claude wins
    claude_wins = (kfold_df['claude_mae'] < kfold_df['ols_mae']).sum()
    claude_wins_lasso = (kfold_df['claude_mae'] < kfold_df['lasso_mae']).sum()
    
    print(f"\nClaude wins vs OLS: {claude_wins}/10 folds ({claude_wins*10}%)")
    print(f"Claude wins vs Lasso: {claude_wins_lasso}/10 folds ({claude_wins_lasso*10}%)")
    
    # Statistical test
    t_stat, p_val = ttest_rel(kfold_df['claude_mae'], kfold_df['ols_mae'])
    print(f"\nPaired t-test (Claude vs OLS): t = {t_stat:.3f}, p = {p_val:.4f}")
    
    t_stat_lasso, p_val_lasso = ttest_rel(kfold_df['claude_mae'], kfold_df['lasso_mae'])
    print(f"Paired t-test (Claude vs Lasso): t = {t_stat_lasso:.3f}, p = {p_val_lasso:.4f}")
    
    # Save results
    kfold_df.to_csv('claude_kfold_stability.csv', index=False)
    print("\n✓ Saved claude_kfold_stability.csv")

# ================================================================
# 5. Subgroup Analysis
# ================================================================

print("\n" + "="*80)
print("5. SUBGROUP ROBUSTNESS ANALYSIS")
print("="*80)
print("Testing if Claude performs consistently across different subgroups")

if 'pred_claude' in analysis_df.columns and 'continent' in analysis_df.columns:
    print("\n5.1 Performance by Continent:")
    print("-" * 60)
    
    continent_results = []
    for continent in sorted(analysis_df['continent'].dropna().unique()):
        subset = analysis_df[analysis_df['continent'] == continent]
        if len(subset) >= 5:  # Only analyze if enough samples
            claude_mae = mean_absolute_error(subset['ground_truth_pi'], subset['pred_claude'])
            
            # Compare to other models
            comparisons = {}
            for model_name, col in models_to_test.items():
                if model_name != 'Claude' and col in subset.columns:
                    model_mae = mean_absolute_error(subset['ground_truth_pi'], subset[col])
                    comparisons[model_name] = model_mae
            
            print(f"\n{continent} (n={len(subset)}):")
            print(f"   Claude MAE: {claude_mae:.2f}pp")
            for model_name, mae in comparisons.items():
                diff = mae - claude_mae
                status = "✓ Claude better" if diff > 0 else "✗ Claude worse"
                print(f"   {model_name} MAE: {mae:.2f}pp (diff: {diff:+.2f}pp) {status}")
            
            continent_results.append({
                'continent': continent,
                'n': len(subset),
                'claude_mae': claude_mae
            })
    
    continent_df = pd.DataFrame(continent_results)
    continent_df.to_csv('claude_continent_performance.csv', index=False)
    print("\n✓ Saved claude_continent_performance.csv")

# Quartile analysis
if 'pred_claude' in analysis_df.columns and 'gdp_capita_2021' in analysis_df.columns:
    print("\n5.2 Performance by GDP Quartile:")
    print("-" * 60)
    
    analysis_df['gdp_quartile'] = pd.qcut(analysis_df['gdp_capita_2021'], 
                                           q=4, labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'])
    
    for quartile in ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']:
        subset = analysis_df[analysis_df['gdp_quartile'] == quartile]
        if len(subset) > 0:
            claude_mae = mean_absolute_error(subset['ground_truth_pi'], subset['pred_claude'])
            print(f"\n{quartile} (n={len(subset)}):")
            print(f"   Claude MAE: {claude_mae:.2f}pp")

# ================================================================
# 6. Error Pattern Analysis
# ================================================================

print("\n" + "="*80)
print("6. ERROR PATTERN ANALYSIS")
print("="*80)
print("Analyzing if Claude has systematic biases or just random errors")

if 'pred_claude' in analysis_df.columns:
    # Calculate errors
    analysis_df['claude_error'] = analysis_df['pred_claude'] - analysis_df['ground_truth_pi']
    analysis_df['claude_abs_error'] = np.abs(analysis_df['claude_error'])
    
    print("\n6.1 Error Distribution:")
    print(f"   Mean Error (bias): {analysis_df['claude_error'].mean():.2f}pp")
    print(f"   Median Error: {analysis_df['claude_error'].median():.2f}pp")
    print(f"   Error Std Dev: {analysis_df['claude_error'].std():.2f}pp")
    print(f"   Skewness: {stats.skew(analysis_df['claude_error']):.3f}")
    print(f"   Kurtosis: {stats.kurtosis(analysis_df['claude_error']):.3f}")
    
    # Test for systematic bias
    t_stat, p_val = stats.ttest_1samp(analysis_df['claude_error'], 0)
    print(f"\n6.2 Systematic Bias Test:")
    print(f"   H0: Mean error = 0 (no systematic bias)")
    print(f"   t = {t_stat:.3f}, p = {p_val:.4f}")
    if p_val < 0.05:
        print(f"   ⚠️  Significant systematic bias detected")
    else:
        print(f"   ✓ No significant systematic bias")
    
    # Identify outliers
    print(f"\n6.3 Outlier Analysis:")
    outlier_threshold = analysis_df['claude_abs_error'].quantile(0.90)
    outliers = analysis_df[analysis_df['claude_abs_error'] > outlier_threshold]
    print(f"   Countries with largest errors (top 10%):")
    for _, row in outliers.nsmallest(10, 'claude_abs_error').iterrows():
        print(f"      {row['countrynew']}: {row['claude_error']:.1f}pp error")

# ================================================================
# 7. Multiple Random Seeds Test
# ================================================================

print("\n" + "="*80)
print("7. CONSISTENCY ACROSS RANDOM SEEDS")
print("="*80)
print("Testing if results hold with different random initializations (20 seeds)")

def evaluate_with_seed(X, y, llm_preds, seed):
    """Evaluate models with a specific random seed"""
    # 80-20 split - need to do this in two steps
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    
    # Split llm_preds using the same indices
    _, llm_test = train_test_split(
        llm_preds, test_size=0.2, random_state=seed
    )
    
    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # OLS
    ols = LinearRegression()
    ols.fit(X_train_scaled, y_train)
    ols_mae = mean_absolute_error(y_test, ols.predict(X_test_scaled))
    
    # Lasso
    lasso = LassoCV(cv=5, random_state=seed)
    lasso.fit(X_train_scaled, y_train)
    lasso_mae = mean_absolute_error(y_test, lasso.predict(X_test_scaled))
    
    # Claude
    claude_mae = mean_absolute_error(y_test, llm_test)
    
    return claude_mae, ols_mae, lasso_mae

if 'pred_claude' in analysis_df.columns and len(available_features) > 0:
    seeds = range(42, 62)  # 20 different seeds
    seed_results = []
    
    X = analysis_df[available_features]
    y = analysis_df['ground_truth_pi']
    llm_preds = analysis_df['pred_claude']
    
    for seed in seeds:
        claude_mae, ols_mae, lasso_mae = evaluate_with_seed(X, y, llm_preds, seed)
        seed_results.append({
            'seed': seed,
            'claude_mae': claude_mae,
            'ols_mae': ols_mae,
            'lasso_mae': lasso_mae,
            'claude_wins_ols': claude_mae < ols_mae,
            'claude_wins_lasso': claude_mae < lasso_mae
        })
    
    seed_df = pd.DataFrame(seed_results)
    
    print(f"\nResults across 20 random seeds:")
    print(f"   Claude MAE: {seed_df['claude_mae'].mean():.2f} ± {seed_df['claude_mae'].std():.2f}pp")
    print(f"   OLS MAE:   {seed_df['ols_mae'].mean():.2f} ± {seed_df['ols_mae'].std():.2f}pp")
    print(f"   Lasso MAE: {seed_df['lasso_mae'].mean():.2f} ± {seed_df['lasso_mae'].std():.2f}pp")
    
    print(f"\nConsistency:")
    claude_win_rate_ols = seed_df['claude_wins_ols'].mean() * 100
    claude_win_rate_lasso = seed_df['claude_wins_lasso'].mean() * 100
    print(f"   Claude beats OLS: {claude_win_rate_ols:.0f}% of seeds")
    print(f"   Claude beats Lasso: {claude_win_rate_lasso:.0f}% of seeds")
    
    if claude_win_rate_ols >= 50:
        print(f"   ✓ Claude consistently competitive with OLS")
    else:
        print(f"   ✗ Claude inconsistently competitive with OLS")
    
    # Save results
    seed_df.to_csv('claude_seed_consistency.csv', index=False)
    print("\n✓ Saved claude_seed_consistency.csv")

# ================================================================
# 8. Create Summary Visualization
# ================================================================

print("\n" + "="*80)
print("8. CREATING VISUALIZATIONS")
print("="*80)

fig = plt.figure()
fig.set_size_inches(18, 12)
fig.set_dpi(300)
matplotlib.rcParams.update({})
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Bootstrap CIs
if bootstrap_results:
    ax1 = fig.add_subplot(gs[0, :2])
    models = list(bootstrap_results.keys())
    means = [bootstrap_results[m]['mean'] for m in models]
    lowers = [bootstrap_results[m]['lower'] for m in models]
    uppers = [bootstrap_results[m]['upper'] for m in models]
    
    ax1.errorbar(models, means, 
                yerr=[np.array(means) - np.array(lowers), 
                      np.array(uppers) - np.array(means)],
                fmt='o', capsize=5, capthick=2, markersize=8)
    
    # Only add Claude reference line if Claude is in results
    if 'Claude' in bootstrap_results:
        ax1.axhline(y=bootstrap_results['Claude']['mean'], 
                   color='red', linestyle='--', alpha=0.5, label='Claude mean')
        ax1.legend()
    
    ax1.set_ylabel('Test MAE (pp)', fontsize=12)
    ax1.set_title('Bootstrap 95% Confidence Intervals', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)

# 2. K-fold stability
if 'kfold_df' in locals():
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.boxplot([kfold_df['claude_mae'], kfold_df['ols_mae'], kfold_df['lasso_mae']],
                labels=['Claude', 'OLS', 'Lasso'])
    ax2.set_ylabel('Test MAE (pp)', fontsize=12)
    ax2.set_title('10-Fold CV Stability', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)

# 3. Error distribution
if 'claude_error' in analysis_df.columns:
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.hist(analysis_df['claude_error'], bins=30, edgecolor='black', alpha=0.7)
    ax3.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No bias')
    ax3.axvline(x=analysis_df['claude_error'].mean(), 
               color='blue', linestyle='-', linewidth=2, label=f'Mean: {analysis_df["claude_error"].mean():.1f}pp')
    ax3.set_xlabel('Prediction Error (pp)', fontsize=12)
    ax3.set_ylabel('Frequency', fontsize=12)
    ax3.set_title('Claude Error Distribution', fontsize=14, fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

# 4. Prediction vs actual
if 'pred_claude' in analysis_df.columns:
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.scatter(analysis_df['ground_truth_pi'], analysis_df['pred_claude'], alpha=0.6)
    ax4.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect prediction')
    ax4.set_xlabel('Ground Truth PI (%)', fontsize=12)
    ax4.set_ylabel('Claude Prediction (%)', fontsize=12)
    ax4.set_title('Claude: Predicted vs Actual', fontsize=14, fontweight='bold')
    
    # Add correlation
    r, p = pearsonr(analysis_df['ground_truth_pi'], analysis_df['pred_claude'])
    ax4.text(0.05, 0.95, f'r = {r:.3f}\np = {p:.4f}', 
            transform=ax4.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax4.legend()
    ax4.grid(True, alpha=0.3)

# 5. Residuals vs predicted
if 'claude_error' in analysis_df.columns:
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.scatter(analysis_df['pred_claude'], analysis_df['claude_error'], alpha=0.6)
    ax5.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax5.set_xlabel('Predicted PI (%)', fontsize=12)
    ax5.set_ylabel('Residual (pp)', fontsize=12)
    ax5.set_title('Residual Plot', fontsize=14, fontweight='bold')
    ax5.grid(True, alpha=0.3)

# 6. Continent performance
if 'continent_df' in locals():
    ax6 = fig.add_subplot(gs[2, 0])
    continent_df = continent_df.sort_values('claude_mae')
    ax6.barh(continent_df['continent'], continent_df['claude_mae'])
    ax6.set_xlabel('MAE (pp)', fontsize=12)
    ax6.set_title('Claude MAE by Continent', fontsize=14, fontweight='bold')
    ax6.grid(True, alpha=0.3, axis='x')

# 7. Seed consistency
if 'seed_df' in locals():
    ax7 = fig.add_subplot(gs[2, 1])
    ax7.scatter(seed_df['seed'], seed_df['claude_mae'], label='Claude', alpha=0.7)
    ax7.scatter(seed_df['seed'], seed_df['ols_mae'], label='OLS', alpha=0.7)
    ax7.scatter(seed_df['seed'], seed_df['lasso_mae'], label='Lasso', alpha=0.7)
    ax7.set_xlabel('Random Seed', fontsize=12)
    ax7.set_ylabel('Test MAE (pp)', fontsize=12)
    ax7.set_title('Consistency Across Seeds', fontsize=14, fontweight='bold')
    ax7.legend()
    ax7.grid(True, alpha=0.3)

# 8. Model ranking frequency
if 'seed_df' in locals():
    ax8 = fig.add_subplot(gs[2, 2])
    rankings = []
    for _, row in seed_df.iterrows():
        maes = [('Claude', row['claude_mae']), ('OLS', row['ols_mae']), ('Lasso', row['lasso_mae'])]
        sorted_models = sorted(maes, key=lambda x: x[1])
        rankings.append(sorted_models[0][0])
    
    from collections import Counter
    rank_counts = Counter(rankings)
    models_ranked = list(rank_counts.keys())
    counts = [rank_counts[m] for m in models_ranked]
    
    ax8.bar(models_ranked, counts)
    ax8.set_ylabel('Frequency (out of 20)', fontsize=12)
    ax8.set_title('Best Model Frequency', fontsize=14, fontweight='bold')
    ax8.grid(True, alpha=0.3, axis='y')

plt.suptitle('CLAUDE ROBUSTNESS ANALYSIS', fontsize=16, fontweight='bold', y=0.995)
plt.savefig('claude_robustness_comprehensive.png', dpi=300, bbox_inches='tight')
plt.savefig('claude_robustness_comprehensive.pdf', bbox_inches='tight')
print("✓ Saved claude_robustness_comprehensive.png")
print("✓ Saved claude_robustness_comprehensive.pdf")
plt.close()

# ================================================================
# 9. Final Summary Report
# ================================================================

print("\n" + "="*80)
print("9. GENERATING SUMMARY REPORT")
print("="*80)

with open('claude_robustness_report.txt', 'w') as f:
    f.write("="*80 + "\n")
    f.write("CLAUDE PERFORMANCE VALIDATION REPORT\n")
    f.write("="*80 + "\n\n")
    
    f.write("QUESTION: Is Claude's superior performance genuine or a statistical fluke?\n\n")
    
    f.write("="*80 + "\n")
    f.write("1. BOOTSTRAP CONFIDENCE INTERVALS\n")
    f.write("="*80 + "\n")
    if bootstrap_results:
        for model, results in bootstrap_results.items():
            f.write(f"\n{model}:\n")
            f.write(f"   MAE: {results['mean']:.2f}pp\n")
            f.write(f"   95% CI: [{results['lower']:.2f}, {results['upper']:.2f}]\n")
        
        if 'Claude' in bootstrap_results:
            f.write("\nCI Overlap Analysis:\n")
            claude_ci = bootstrap_results['Claude']
            for model_name, results in bootstrap_results.items():
                if model_name != 'Claude':
                    overlap = not (results['lower'] > claude_ci['upper'] or 
                                  results['upper'] < claude_ci['lower'])
                    f.write(f"   Claude vs {model_name}: {'OVERLAPPING' if overlap else 'NO OVERLAP'}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("2. K-FOLD CROSS-VALIDATION\n")
    f.write("="*80 + "\n")
    if 'kfold_df' in locals():
        f.write(f"\nMean MAE across 10 folds:\n")
        f.write(f"   Claude:  {kfold_df['claude_mae'].mean():.2f} ± {kfold_df['claude_mae'].std():.2f}pp\n")
        f.write(f"   OLS:    {kfold_df['ols_mae'].mean():.2f} ± {kfold_df['ols_mae'].std():.2f}pp\n")
        f.write(f"   Lasso:  {kfold_df['lasso_mae'].mean():.2f} ± {kfold_df['lasso_mae'].std():.2f}pp\n")
        
        claude_wins = (kfold_df['claude_mae'] < kfold_df['ols_mae']).sum()
        claude_wins_lasso = (kfold_df['claude_mae'] < kfold_df['lasso_mae']).sum()
        f.write(f"\nClaude wins vs OLS: {claude_wins}/10 folds\n")
        f.write(f"Claude wins vs Lasso: {claude_wins_lasso}/10 folds\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("3. RANDOM SEED CONSISTENCY\n")
    f.write("="*80 + "\n")
    if 'seed_df' in locals():
        f.write(f"\nResults across 20 random seeds:\n")
        f.write(f"   Claude MAE: {seed_df['claude_mae'].mean():.2f} ± {seed_df['claude_mae'].std():.2f}pp\n")
        f.write(f"   OLS MAE:   {seed_df['ols_mae'].mean():.2f} ± {seed_df['ols_mae'].std():.2f}pp\n")
        f.write(f"   Lasso MAE: {seed_df['lasso_mae'].mean():.2f} ± {seed_df['lasso_mae'].std():.2f}pp\n")
        
        claude_win_rate_ols = seed_df['claude_wins_ols'].mean() * 100
        claude_win_rate_lasso = seed_df['claude_wins_lasso'].mean() * 100
        f.write(f"\n   Claude beats OLS: {claude_win_rate_ols:.0f}% of seeds\n")
        f.write(f"   Claude beats Lasso: {claude_win_rate_lasso:.0f}% of seeds\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("4. ERROR PATTERN ANALYSIS\n")
    f.write("="*80 + "\n")
    if 'claude_error' in analysis_df.columns:
        f.write(f"\nError Statistics:\n")
        f.write(f"   Mean Error (bias): {analysis_df['claude_error'].mean():.2f}pp\n")
        f.write(f"   Std Dev: {analysis_df['claude_error'].std():.2f}pp\n")
        f.write(f"   Skewness: {stats.skew(analysis_df['claude_error']):.3f}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("CONCLUSION\n")
    f.write("="*80 + "\n")
    f.write("\nBased on comprehensive robustness testing:\n\n")
    
    # Automated conclusion
    conclusion_points = []
    
    if 'kfold_df' in locals():
        claude_wins = (kfold_df['claude_mae'] < kfold_df['ols_mae']).sum()
        if claude_wins >= 5:
            conclusion_points.append("✓ Claude performance is CONSISTENT across k-fold validation")
        else:
            conclusion_points.append("✗ Claude performance is INCONSISTENT across k-fold validation")
    
    if 'seed_df' in locals():
        claude_win_rate = seed_df['claude_wins_ols'].mean() * 100
        if claude_win_rate >= 50:
            conclusion_points.append("✓ Claude performance is ROBUST across random seeds")
        else:
            conclusion_points.append("✗ Claude performance is SENSITIVE to random initialization")
    
    if bootstrap_results and 'Claude' in bootstrap_results:
        claude_ci = bootstrap_results['Claude']
        ols_better = False
        if 'OLS' in bootstrap_results:
            ols_ci = bootstrap_results['OLS']
            if ols_ci['upper'] < claude_ci['lower']:
                ols_better = True
        
        if not ols_better:
            conclusion_points.append("✓ Claude performance is STATISTICALLY COMPETITIVE")
        else:
            conclusion_points.append("✗ OLS significantly outperforms Claude")
    
    for point in conclusion_points:
        f.write(point + "\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("FILES GENERATED:\n")
    f.write("   - claude_robustness_report.txt (this file)\n")
    f.write("   - claude_robustness_comprehensive.png (visualization)\n")
    f.write("   - claude_robustness_comprehensive.pdf (visualization)\n")
    f.write("   - claude_kfold_stability.csv\n")
    f.write("   - claude_seed_consistency.csv\n")
    f.write("   - claude_continent_performance.csv\n")

print("✓ Saved claude_robustness_report.txt")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nKey files created:")
print("   1. claude_robustness_report.txt - Comprehensive text summary")
print("   2. claude_robustness_comprehensive.png - 8-panel visualization")
print("   3. claude_robustness_comprehensive.pdf - 8-panel visualization (PDF)")
print("   4. claude_kfold_stability.csv - K-fold CV results")
print("   5. claude_seed_consistency.csv - Random seed test results")
print("   6. claude_continent_performance.csv - Subgroup analysis")
print("\n" + "="*80)

CLAUDE PERFORMANCE VALIDATION & ROBUSTNESS TESTING

1. LOADING DATA
✓ Loaded 1000 predictions
✓ Countries: 125
✓ Ground truth available: 1000 observations

Before dropna: 125 rows
Columns: ['countrynew', 'ground_truth_pi', 'continent', 'mean_age', 'mean_edu', 'mean_religion', 'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth', 'hdi_2021', 'pred_claude', 'pred_gpt', 'pred_llama', 'pred_gemini', 'pred_ensemble']
Missing values per column:
countrynew         0
ground_truth_pi    0
continent          0
mean_age           3
mean_edu           3
mean_religion      9
gdp_capita_2021    0
top1pct_income     0
top1pct_wealth     0
hdi_2021           2
pred_claude        0
pred_gpt           0
pred_llama         0
pred_gemini        0
pred_ensemble      0
dtype: int64

✓ Complete cases for analysis: 114

2. BOOTSTRAP CONFIDENCE INTERVALS
Testing if confidence intervals overlap (1000 bootstrap samples)
Checking Claude (pred_claude)...

Claude:
   MAE: 4.88pp
   95% CI: [4.15, 5.65]
   CI Width

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>